# Разведочный анализ данных перед логистической регрессией

Цель — исследовать **все 14 полей**, зафиксировать ограничения данных и подготовить решения о признаках. Логистическая регрессия выбрана для следующего этапа; в этой тетради модели не обучаются.

Основной CSV сохраняется без изменений. После технической проверки и разбиения анализ связей с целями выполняется только на обучающей выборке. Тестовая выборка остаётся для итоговой оценки.

**Статус при создании:** тетрадь подготовлена, но ещё не выполнена: зависимости Python 3.11 будут установлены позже по решению пользователя. Пустые результаты не означают отсутствие проблем в данных.

Работайте по ячейкам; дополняйте блоки «Ваши наблюдения». Не устанавливайте библиотеки через %pip: используйте uv add и uv sync из корня проекта. См. ../README.md и ../TERMINOLOGY.md.

## 1. Окружение и конфигурация

Проверяем, что ядро принадлежит .venv проекта. Настройки путей, случайности, разбиения и статуса доступности полей находятся в configs/eda.json. Все имена исходных полей остаются английскими.

In [ ]:
import json
import sys
from pathlib import Path
from importlib.metadata import version

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'configs/eda.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Откройте тетрадь из каталога проекта.')
assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve(), 'Выберите ядро Python 3.11 (.venv, кейс 1).'
assert sys.version_info[:2] == (3, 11), 'Нужен Python 3.11.'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from industrial_maintenance import data, eda, plots

%matplotlib inline
plots.setup_style()
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 180)
CONFIG = data.load_config(ROOT)
OUTPUT = ROOT / 'artifacts/eda' / CONFIG['split_version']
FIGURES = OUTPUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], '|', sys.executable)
display(pd.Series({name: version(name) for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn']}, name='Версия'))
display(CONFIG)

## 2. Первичная техническая проверка всего CSV

Здесь проверяются происхождение, контрольная сумма, схема, типы, пропуски, id и повторы. Полные данные нужны для защиты границ выборок; связи с целями до разбиения не используются для отбора признаков.

In [ ]:
df, source_manifest = data.load_dataset(ROOT, CONFIG)
display(pd.Series(source_manifest, name='Источник'))
raw_quality = eda.quality_table(df)
display(raw_quality)
display(data.duplicate_audit(df))

Повторение значения equipment или категории цели само по себе не является дубликатом записи. Повторение operator_log важно для сравнения текстовых моделей. Группы одинаковых журналов не являются доказанными идентификаторами физической машины.

**Ваши наблюдения:**
- Какие поля требуют уточнения определения или единиц?
- Есть ли пропуски/повторы, которые меняют план исследования?

## 3. Разбиение до исследования связей

Одинаковые operator_log после приведения регистра и пробелов помещаются в одну группу. Семь частей распределяются как пять обучающих, одна валидационная и одна тестовая; доли приблизительные. Используется приблизительная стратификация по паре fault_type и failure_within_30d.

Это обоснованное начальное правило на основании повторяемости журналов, а не временное разбиение. Оно не устраняет сходство текстов с разными числами. Его ограничение показывается отдельно; выводы о новых шаблонах/машинах/времени требуют другого заранее определённого эксперимента.

Первый запуск сохраняет принадлежность id и протокол. Повторный использует те же файлы, проверяет их контрольную сумму и запрещает молчаливое изменение.

In [ ]:
membership, split_manifest = data.get_or_create_splits(ROOT, df, CONFIG)
display(membership.groupby('split').size().rename('Число записей'))
overlap = data.split_overlap_audit(df, membership)
display(overlap)
print(split_manifest['limitation'])
train_ids = set(membership.loc[membership['split'] == 'train', 'id'])
train = df.loc[df['id'].isin(train_ids)].copy().reset_index(drop=True)
assert len(train) > 0
print('Для дальнейшего EDA используются только train:', len(train))

**Ваши наблюдения:**
- Совпадают ли точные тексты между частями? Это должно быть исключено проверками.
- Какое сходство остаётся после маскирования чисел?
- Какой сценарий применения проверяет такое разбиение, а какой не проверяет?

Не открывайте тестовые значения для настройки признаков, правил обработки или порогов.

## 4. Цели и категории: обучающая выборка

Исследуем обе задачи и обязательные подгруппы. Баланс классов влияет на выбор метрик и дальнейшую проверку взвешивания классов, но не является основанием менять тестовую выборку.

In [ ]:
for field in ['failure_within_30d', 'fault_type', 'equipment', 'severity']:
    print(field)
    display(eda.class_balance(train, field))
print('Совместное распределение целей на train')
display(pd.crosstab(train['fault_type'], train['failure_within_30d']))
display(train.sample(n=min(8, len(train)), random_state=CONFIG['seed']))

**Ваши наблюдения:**
- Какие классы представлены слабее остальных?
- Совпадает ли fault_type с событием отказа? Не делайте такого предположения заранее.
- Достаточно ли примеров для анализа каждой подгруппы?

## 5. Числовые поля: распределения, пропуски, выбросы

Анализ включает пять показаний **и horizon_days**. Последнее поле исследуется, хотя его доступность до прогноза пока неизвестна.

IQR используется как описательное правило для отметки крайних значений, а не как автоматический критерий удаления. Физически допустимые границы требуют технического описания. Температура вне обычного диапазона может быть полезным признаком неисправности, а не ошибкой.

In [ ]:
numeric = CONFIG['numeric_columns']
numeric_statistics = eda.numeric_summary(train, numeric)
outliers = eda.outlier_table(train, numeric)
display(numeric_statistics)
display(outliers)
figure = plots.numerical_distributions(train, numeric)
figure.savefig(FIGURES / 'numeric_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
figure = plots.numeric_by_target(train, numeric, 'failure_within_30d')
figure.savefig(FIGURES / 'numeric_by_failure.png', bbox_inches='tight')
plt.show()
figure = plots.numeric_by_target(train, numeric, 'fault_type')
figure.savefig(FIGURES / 'numeric_by_fault.png', bbox_inches='tight')
plt.show()

**Ваши наблюдения:**
- Какие диапазоны выглядят необычно и почему?
- Для удаления значения есть подтверждённая ошибка измерения или только статистическая необычность?
- Меняются ли распределения между классами? Важны и величина различий, и перекрытие распределений.

## 6. Связи между числовыми полями

Spearman correlation — связь рангов значений, полезная для монотонных зависимостей. Она не устанавливает причинность и не проверяет все возможные нелинейные взаимодействия.

Длина operator_log здесь — диагностическое производное поле, а не добавленный столбец исходного CSV и не утверждённый вход модели. Сильная связь двух признаков означает необходимость проверить повторение информации и устойчивость модели; не означает автоматического удаления одного из них.

In [ ]:
diagnostic = train.assign(operator_log_length=train['operator_log'].fillna('').str.len())
correlations = diagnostic[numeric + ['operator_log_length']].corr(method='spearman')
display(correlations)
figure = plots.correlation_plot(diagnostic, numeric + ['operator_log_length'])
figure.savefig(FIGURES / 'numeric_correlations.png', bbox_inches='tight')
plt.show()

## 7. Категориальные поля и отношения между полями

Не исключаем equipment и manual_id из рассмотрения из-за их категориального типа. Проверяем, сколько информации они несут, повторяют ли друг друга и что обозначают.

severity и required_route тоже присутствуют в анализе. Их связь с целями может объясняться получением после диагностики. Пользователь подтвердил, что момент доступности пока неизвестен.

In [ ]:
figure = plots.categorical_panels(train, CONFIG['categorical_columns'])
figure.savefig(FIGURES / 'categorical_distributions.png', bbox_inches='tight')
plt.show()
relationships = {}
for left, right in [('equipment', 'manual_id'), ('fault_type', 'severity'), ('fault_type', 'required_route'), ('severity', 'failure_within_30d')]:
    result = eda.field_relationships(train, left, right)
    relationships[f'{left}__{right}'] = result['table']
    display(result['table'])
    display(result['summary'])

**Ваши наблюдения:**
- Есть ли однозначное соответствие equipment и manual_id?
- Если два поля повторяют информацию, в каком контролируемом сравнении это проверить?
- Можно ли объяснить сильную связь с целью реальным порядком получения поля? Пока нет — статус остаётся открытым.

## 8. Связи каждого кандидата с целями

Методы приводятся с английскими названиями до согласования новых терминов в TERMINOLOGY.md:

- Mann–Whitney U: сравнение двух групп; rank-biserial показывает направленную величину различия в рангах.
- Kruskal–Wallis: сравнение нескольких групп; epsilon-squared характеризует различия между группами.
- Chi-square и corrected Cramér's V: связь категориальных полей; отдельно отмечаются малые ожидаемые числа в таблицах.
- p-value не показывает полезность признака или вероятность правильности гипотезы. q-value (BH) корректирует семейство проверок отдельно для каждой цели.

Эти сравнения описательные: независимость событий не подтверждена, повторяемость текстов может уменьшать эффективное число независимых наблюдений. Сравнивать числовые значения разных мер эффекта напрямую нельзя. Отбор по p-value/q-value автоматически не выполняется.

При большой выборке очень малое различие может дать малое p-value. Несущественная связь по одному полю не исключает его пользы совместно с другими. См. документацию [SciPy](https://docs.scipy.org/doc/scipy/reference/stats.html) и [коррекции BH](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html).

In [ ]:
associations = eda.association_table(train, CONFIG)
display(associations)
print('Результаты не являются списком выбранных признаков.')

**Ваши наблюдения:**
- Какие различия достаточно велики, чтобы иметь практический смысл?
- Где p-value ненадёжен из-за малых ожидаемых чисел или зависимых записей?
- Для какого поля нужна предметная проверка доступности, даже если связь очень сильная?

## 9. Подгруппы и неизвестная доступность severity

Смотрим наблюдаемую долю failure_within_30d в equipment и severity с числом записей в каждой группе. Различия не доказывают причинность и не разрешают использовать severity до прогноза.

In [ ]:
subgroups = {}
for field in ['equipment', 'severity']:
    figure, table = plots.failure_by_category(train, field)
    subgroups[field] = table
    display(table)
    figure.savefig(FIGURES / f'failure_by_{field}.png', bbox_inches='tight')
    plt.show()

## 10. operator_log: содержание, повторяемость и повторение показаний

Исследуем текст, даже если первый контроль A по заданию использует только числовые поля. Из ограничения конкретного эксперимента нельзя делать вывод, что текст бесполезен.

Слова, связанные с классом, могут быть законными наблюдаемыми симптомами или раскрывать уже поставленный диагноз. Различить эти случаи без информации о порядке формирования журнала нельзя.

Упомянутая в тексте нагрузка сравнивается с load_ratio. Расхождение до округления или повторение числа не доказывает независимое измерение.

In [ ]:
text_statistics, load_check = eda.text_audit(train)
display(text_statistics)
display(load_check['Разница, процентные пункты'].describe())
display(load_check.dropna().sample(n=min(10, load_check.dropna().shape[0]), random_state=CONFIG['seed']))
masked_texts = train['operator_log'].map(lambda text: data.normalize_log(text, mask_numbers=True))
display(masked_texts.value_counts().rename_axis('Текст после маскирования чисел').to_frame('Число на train').head(20))

In [ ]:
tokens = {}
for target in CONFIG['targets']:
    print(target)
    tokens[target] = eda.token_associations(train, target, CONFIG['min_token_documents'])
    display(tokens[target])

**Ваши наблюдения:**
- Есть ли явные диагнозы/сведения об исходе в тексте?
- Какие слова обозначают симптомы и доступны до решения?
- Можно ли считать журналы независимыми или они отражают небольшой набор повторяющихся формулировок?
- Что покажет классическая текстовая модель по сравнению с трансформером? Это проверяется позже, а не выводится из частот слов.

## 11. id как идентификатор, а не время

Проверяем, нет ли явной связи порядка id с составом данных. Даже при такой связи id не становится показанием оборудования. Без определения источника его нельзя трактовать как временную метку.

Производное id_bin ниже используется только для диагностики порядка. Исходный id не переименовывается.

In [ ]:
id_numeric = pd.to_numeric(train['id'], errors='coerce')
if id_numeric.notna().all() and id_numeric.nunique() >= 10:
    order_audit = train.assign(id_bin=pd.qcut(id_numeric, q=10, duplicates='drop'))
    display(order_audit.groupby('id_bin', observed=True)['failure_within_30d'].agg(['size', 'mean']))
else:
    print('id не подходит для числовой диагностики порядка; это не повод преобразовывать его во вход модели.')

## 12. Обоснование для каждого поля

В таблице есть **все исходные поля**, статистические сведения, роль, доступность и следующая проверка. Ни один кандидат не исключается только по корреляции или названию.

- id имеет роль идентификатора, а обе цели остаются целевыми переменными.
- Пять числовых показаний — кандидаты контроля A по формулировке задания, но окончательная предобработка и состав ещё обсуждаются.
- Для equipment и manual_id нужны смысл и проверки повторения информации.
- operator_log остаётся для текстового контроля, B и C; числовой контроль A ограничен по определению эксперимента.
- severity, required_route и horizon_days ждут описания данных. До подтверждения не считать их доступными входами.

Заполните «Решение команды» и «Источник решения» после анализа. Источником может быть описание данных, наблюдение в этой тетради или последующий контролируемый эксперимент на валидационной выборке.

In [ ]:
ledger = eda.feature_ledger(train, associations, CONFIG)
assert set(ledger['Поле']) == set(df.columns)
display(ledger)

## 13. Сохранение материалов и ваших решений

Повторно вычисляемые факты сохраняются отдельно от редактируемого файла решений. Повторный запуск не стирает feature_decisions.csv. Если уже есть решения, новая статистическая основа записывается в feature_evidence_current.csv.

Исходный CSV не меняется. Ни прогнозов модели, ни итоговых тестовых метрик на этом этапе нет.

In [ ]:
tables = {
    'quality_all_records': raw_quality,
    'split_overlap': overlap,
    'train_numeric_summary': numeric_statistics,
    'train_outlier_flags': outliers,
    'train_associations': associations,
    'train_correlations': correlations,
    'train_text_summary': text_statistics,
    'feature_decisions': ledger,
}
for key, table in relationships.items():
    tables[f'train_relation_{key}'] = table
for target, table in tokens.items():
    tables[f'train_tokens_{target}'] = table
for field, table in subgroups.items():
    tables[f'train_subgroup_{field}'] = table
run_manifest = {
    'source': source_manifest, 'split': split_manifest,
    'analysis_partition': 'train', 'analysis_rows': len(train),
    'config': CONFIG,
    'python': sys.version,
    'libraries': {name: version(name) for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn']},
    'feature_selection_completed': False, 'models_trained': False,
}
eda.export_tables(OUTPUT, tables, run_manifest)
assert data.file_sha256(ROOT / CONFIG['dataset']) == source_manifest['sha256']
print('Материалы:', OUTPUT.relative_to(ROOT))
display(pd.read_csv(OUTPUT / 'feature_decisions.csv', encoding='utf-8-sig').drop(columns=['Unnamed: 0'], errors='ignore'))

## 14. Ваш дополнительный анализ

Добавляйте свои ячейки здесь и используйте train. Если требуется операция на validation, заранее сформулируйте проверяемый вариант; не используйте test для поиска лучшего состава признаков. Настройку моделей и предобработки вынесем в следующий этап.

**Ваши наблюдения:**
- Какие решения уже подтверждены и чем?
- Какие решения ждут описания данных?
- Какие варианты следует сравнить при реализации логистической регрессии?

In [ ]:
# Здесь можно добавить собственный анализ обучающей выборки.
# Пример: train.groupby('equipment')[CONFIG['numeric_columns']].median()


## 15. Готовность к логистической регрессии

Перед реализацией:

1. Проверить результаты EDA и ограничения разбиения; сохранить постоянную тестовую выборку.
2. Для каждого входа зафиксировать смысл, доступность и обоснование в таблице решений.
3. Определить числовой контроль A и дополнительные варианты, например A + equipment, если доступность подтверждена.
4. Описать обработку пропусков, масштабирование и кодирование категорий; все обучаемые операции выполняются только на train.
5. Обе задачи моделировать отдельно или явно обосновать общую архитектуру; не передавать одну эталонную цель как вход для другой.
6. Выбирать настройки и пороги по validation; затем сравнивать на одном test по метрикам задания.

**Решение команды по итогам этой тетради:** пока не принято. Не выдавать подготовленный код и пустые результаты за выполненное исследование.